VARIABLES
<br>
Outcome variable = count_usa<br>
Treatment variables = treat and count_for (pre-treatment treat = 0, post-treatmen treat = 1)<br>
Dummy time variable = grntyr<br>

METHOD<br>
Stata data set is loaded: chem_patents_maindataset.dta, dummy variable for time are created, then the exog and ednog variables are created for the PanelOLS. Then we run the model and print the results.<br>
PanelOLS is used instead of regular OLS because array gets too big and kept getting the Memory allocation error.<br>

RESULTS<br>
                effect    std.error<br>
treat          0.1505     0.0356<br>
The results match both Mora's and original papers results(Table 2 in both papers).


In [3]:
#DiD no time windows

import pandas as pd
from linearmodels.panel import PanelOLS

# Load data

df = pd.read_stata("chem_patents_maindataset.dta")

# Treatment × post indicator

df['postTWEA'] = (df['grntyr'] > 1918).astype(int)
df['treat_post'] = df['treat'] * df['postTWEA']

# Set multi-index (class and year)

df = df.set_index(['class_id', 'grntyr'])

# Regression variables

exog_vars = ['treat_post', 'count_for']
exog = df[exog_vars]
endog = df['count_usa']

# Estimate DID with class FE and year FE

model = PanelOLS(endog, exog, entity_effects=True, time_effects=True)
results = model.fit(cov_type='clustered', cluster_entity=True)

print(results.summary)


                          PanelOLS Estimation Summary                           
Dep. Variable:              count_usa   R-squared:                        0.0203
Estimator:                   PanelOLS   R-squared (Between):              0.0819
No. Observations:              471120   R-squared (Within):               0.0332
Date:                Tue, Jun 02 2026   R-squared (Overall):              0.0531
Time:                        23:16:45   Log-likelihood                -6.105e+05
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      4810.4
Entities:                        7248   P-value                           0.0000
Avg Obs:                       65.000   Distribution:                F(2,463806)
Min Obs:                       65.000                                           
Max Obs:                       65.000   F-statistic (robust):             155.67
                            

In [4]:
# Cross moment without time windows

import pandas as pd
import numpy as np

# Load data 
df = pd.read_stata("chem_patents_maindataset.dta")

# Define post indicator
df['postTWEA'] = (df['grntyr'] > 1918).astype(int)

# Split full sample into pre and post 
data_pre = df[df['postTWEA'] == 0].copy()
data_post = df[df['postTWEA'] == 1].copy()

# Define variables 
D_pre = data_pre['treat'].to_numpy().astype(float)
D_post = data_post['treat'].to_numpy().astype(float)

Z = D_pre.copy()                # pre-treatment D
Y = data_post['count_usa'].to_numpy().astype(float)
D = D_post.copy()

# Ensure equal length
min_len = min(len(Z), len(D), len(Y))
Z = Z[:min_len]
D = D[:min_len]
Y = Y[:min_len]

# Center variables
Z -= np.mean(Z)
D -= np.mean(D)
Y -= np.mean(Y)

# Cross-moment functions
def compute_ratio(Z, D, deg=2):
    var_u = np.mean(Z * D)
    sign = np.sign(var_u)
    diff_normal_D = np.mean(D ** deg * Z) - deg * var_u * np.mean(D ** (deg - 1))
    diff_normal_Z = np.mean(Z ** deg * D) - deg * var_u * np.mean(Z ** (deg - 1))
    epsilon = 1e-8
    alpha_sq = diff_normal_D / (diff_normal_Z + epsilon)
    if alpha_sq < 0:
        alpha_sq = -(abs(alpha_sq) ** (1 / (deg - 1)))
    else:
        alpha_sq = alpha_sq ** (1 / (deg - 1))
    alpha_sq = abs(alpha_sq) * sign
    return alpha_sq

def get_beta(Z, D, Y, deg=2):
    denominator = 0
    while denominator == 0:
        alpha_sq = compute_ratio(Z, D, deg)
        numerator = np.mean(D * Y) - alpha_sq * np.mean(Y * Z)
        denominator = np.mean(D * D) - alpha_sq * np.mean(D * Z)
        deg += 1
        if deg > 20:
            raise ValueError("Cannot compute beta, check data")
    return numerator / denominator

beta = get_beta(Z, D, Y)
print(f"Beta estimated by cross-moment method (full sample): {beta:.4f}")

Beta estimated by cross-moment method (full sample): 0.1102
